In [2]:
!pip install -q delta-spark pyspark

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 kB 1.7 MB/s eta 0:00:00


In [3]:
import delta
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

builder = SparkSession.builder \
    .appName("Part_4_Delta_advanced") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

spark = configure_spark_with_delta_pip(builder).getOrCreate()
print(delta.__version__)
print("Spark session ready")


4.2.0
Spark session ready


In [12]:
data = [
(101,"Arjun Reddy","Hyderabad","Cardiology",5000,1),
(102,"Sneha Kapoor","Delhi","Orthopedics",3000,2),
(103,"Rahul Sharma","Mumbai","Dermatology",1500,1),
(104,"Priya Nair","Bangalore","Cardiology",5000,2),
(105,"Vikram Singh","Chennai","Neurology",7000,1),
(106,"Ananya Das","Kolkata","Orthopedics",3000,3),
(107,"Karan Patel","Ahmedabad","Cardiology",5000,1),
(108,"Meera Iyer","Bangalore","Dermatology",1500,2)
]
columns = [
"visit_id",
"patient_name",
"city",
"department",
"consultation_fee",
"tests_count"
]

df = spark.createDataFrame(data, columns)
df.show()
df.write.format("delta").mode("overwrite").save("/tmp/patient_visits_delta")

+--------+------------+---------+-----------+----------------+-----------+
|visit_id|patient_name|     city| department|consultation_fee|tests_count|
+--------+------------+---------+-----------+----------------+-----------+
|     101| Arjun Reddy|Hyderabad| Cardiology|            5000|          1|
|     102|Sneha Kapoor|    Delhi|Orthopedics|            3000|          2|
|     103|Rahul Sharma|   Mumbai|Dermatology|            1500|          1|
|     104|  Priya Nair|Bangalore| Cardiology|            5000|          2|
|     105|Vikram Singh|  Chennai|  Neurology|            7000|          1|
|     106|  Ananya Das|  Kolkata|Orthopedics|            3000|          3|
|     107| Karan Patel|Ahmedabad| Cardiology|            5000|          1|
|     108|  Meera Iyer|Bangalore|Dermatology|            1500|          2|
+--------+------------+---------+-----------+----------------+-----------+



In [4]:
spark.sql("""
CREATE TABLE patient_visits
USING DELTA
LOCATION '/tmp/patient_visits_delta'
""")

DataFrame[]

In [5]:
spark.sql(""" describe history patient_visits;""").show()

+-------+--------------------+------+--------+------------+--------------------+----+--------+---------+-----------+--------------+-------------+----------------+------------+--------------------+
|version|           timestamp|userId|userName|   operation| operationParameters| job|notebook|clusterId|readVersion|isolationLevel|isBlindAppend|operationMetrics|userMetadata|          engineInfo|
+-------+--------------------+------+--------+------------+--------------------+----+--------+---------+-----------+--------------+-------------+----------------+------------+--------------------+
|      0|2026-05-04 16:41:...|  NULL|    NULL|CREATE TABLE|{partitionBy -> [...|NULL|    NULL|     NULL|       NULL|  Serializable|         true|              {}|        NULL|Apache-Spark/4.0....|
+-------+--------------------+------+--------+------------+--------------------+----+--------+---------+-----------+--------------+-------------+----------------+------------+--------------------+



In [ ]:
spark.sql(""" select * from patient_visits version as of 0 """).show()

In [ ]:
spark.sql(" VACUUM patient_visits RETAIN 168 HOURS DRY RUN;").show()